# XAI Sensitivity & Stability (L4 GPU)

**런타임: L4 GPU** (런타임 → 런타임 유형 변경 → L4)

### 업로드할 파일 (직접 업로드)
- `fireimage_clean.zip`  (약 32MB)
- `weights_E_min.zip`  (약 184MB — fold0 efficientnetv2+maxvit만)

셀 순서대로 실행. 업로드는 **각각 따로**.

In [ ]:
# 1) 클론 + 패키지 (shap 제거 → 설치 빠름)
%cd /content
!rm -rf fireimage_detection
!git clone https://github.com/yuntaewon812/fireimage_detection.git fireimage_detection -q
!pip install timm grad-cam lime scikit-image -q
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# 2-A) 데이터 업로드 — fireimage_clean.zip (약 32MB) 하나만 선택
from google.colab import files
up1 = files.upload()
print('업로드:', list(up1.keys()))

In [ ]:
# 2-B) 가중치 업로드 — weights_E_min.zip (약 184MB) 하나만 선택
#  (끊기면 이 셀만 다시 실행)
from google.colab import files
up2 = files.upload()
print('업로드:', list(up2.keys()))

In [ ]:
# 3) 압축 해제
import zipfile, os
BASE = '/content/fireimage_detection'

if os.path.exists('/content/fireimage_clean.zip'):
    !python {BASE}/colab_setup.py
    print('데이터 압축 해제 완료')
else:
    print('주의: fireimage_clean.zip 없음 — 셀 2-A 다시 실행')

# weights_E_min.zip 또는 weights_E.zip 둘 다 허용
wzip = None
for cand in ['/content/weights_E_min.zip', '/content/weights_E.zip']:
    if os.path.exists(cand):
        wzip = cand; break
if wzip:
    with zipfile.ZipFile(wzip, 'r') as z:
        z.extractall(BASE)
    print('가중치 압축 해제 완료:', os.path.basename(wzip))
else:
    print('주의: 가중치 zip 없음 — 셀 2-B 다시 실행')

w_dir = f'{BASE}/model_save/fireimage_abl_E/fold0'
print('fold0 가중치:', os.listdir(w_dir) if os.path.exists(w_dir) else '없음')

In [ ]:
# 4) Sensitivity & Stability 실행 (L4 GPU)
%cd /content/fireimage_detection
import time
t0 = time.time()
!python sensitivity_stability.py
gpu_total = time.time() - t0
print(f'\n총 GPU 실행 시간: {gpu_total:.1f}s ({gpu_total/60:.1f}분)')

In [ ]:
# 5) 결과 표시
import pandas as pd
gpu_df = pd.read_csv('results_SENS_STAB/sens_stab_cuda.csv')
print('=== GPU (L4) Sensitivity & Stability ===')
print(gpu_df[['Model','Method','Sensitivity','Stability','Time_sec']].to_string(index=False))

In [ ]:
# 6) 결과 다운로드
from google.colab import files
files.download('results_SENS_STAB/sens_stab_cuda.csv')